# Setup

In [1]:
import torch

tensors_dir = "../data/tensors"

X_trainval = torch.load(f"{tensors_dir}/X_trainval.pt").float()
X_test = torch.load(f"{tensors_dir}/X_test.pt").float()
y_trainval = torch.load(f"{tensors_dir}/y_trainval.pt").long()
y_test = torch.load(f"{tensors_dir}/y_test.pt").long()
sample_weights = torch.load(f"{tensors_dir}/sample_weights.pt").float()

print(f"X_train: {X_trainval.shape}  {X_trainval.dtype}")
print(f"X_test: {X_test.shape}   {X_test.dtype}")
print(f"y_train: {y_trainval.shape}  {y_trainval.dtype}")
print(f"y_test: {y_test.shape}   {y_test.dtype}")
print(f"weights: {sample_weights.shape}")

X_train: torch.Size([796, 10])  torch.float32
X_test: torch.Size([199, 10])   torch.float32
y_train: torch.Size([796])  torch.int64
y_test: torch.Size([199])   torch.int64
weights: torch.Size([796])


In [2]:
from src import MLP

INPUT_SIZE = X_trainval.shape[1]
NUM_CLASSES = 4

model = MLP(input_size = INPUT_SIZE, hidden_layers=[16, 64, 256, 512, 256, 64, 16],
            output_size = NUM_CLASSES, dropout_p = 0.3, activation='relu')
print(model)

dummy_output = model(torch.randn(8, INPUT_SIZE))
print(f"\noutput shape: {dummy_output.shape}")
print(f"\noutput values: {dummy_output}")

MLP(
  (input): Linear(in_features=10, out_features=16, bias=True)
  (hidden_layers): ModuleList(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): Linear(in_features=64, out_features=256, bias=True)
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): Linear(in_features=256, out_features=64, bias=True)
    (5): Linear(in_features=64, out_features=16, bias=True)
  )
  (output): Linear(in_features=16, out_features=4, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (activation): ReLU()
)

output shape: torch.Size([8, 4])

output values: tensor([[ 0.1066,  0.2111, -0.1218,  0.1839],
        [ 0.1317,  0.2147, -0.1071,  0.1456],
        [ 0.1269,  0.1664, -0.0743,  0.1313],
        [ 0.1369,  0.2430, -0.1431,  0.1824],
        [ 0.1052,  0.2248, -0.1251,  0.1852],
        [ 0.1090,  0.1922, -0.0934,  0.1503],
        [ 0.1366,  0.1996, -0.1109,  0.1503],
        [ 0.1037,  0.1817, -0.

In [3]:
from src import build_fold_dataloaders

train_loader, val_loader = build_fold_dataloaders(X=X_trainval, y=y_trainval,
                                                  weights=sample_weights, fold_idx=0, batch_size=32, n_splits=5)

X_example_batch, y_example_batch = next(iter(train_loader))
print(f"Test loaders length: \nTrain: {len(train_loader)}   Val: {len(val_loader)}")
print(f"\nX batch info: \nshape: {X_example_batch.shape}   dtype: {X_example_batch.dtype}")
print(f"\nY batch info: \nshape: {y_example_batch.shape}   dtype: {y_example_batch.dtype}")

Test loaders length: 
Train: 20   Val: 5

X batch info: 
shape: torch.Size([32, 10])   dtype: torch.float32

Y batch info: 
shape: torch.Size([32])   dtype: torch.int64


In [4]:
from src import train_fold
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = train_fold(model, train_loader, val_loader, optimizer, criterion, fold_idx=0,
          device=device, n_epochs=10,  write_model=True)
history

[Fold 0] Epoch   0/10 | train: 1.3891 | val: 1.4202 <- new best
[Fold 0] Epoch   1/10 | train: 1.3799 | val: 1.3682 <- new best
[Fold 0] Epoch   2/10 | train: 1.3621 | val: 1.2960 <- new best
[Fold 0] Epoch   3/10 | train: 1.3320 | val: 1.3173
[Fold 0] Epoch   4/10 | train: 1.3026 | val: 1.3207
[Fold 0] Epoch   5/10 | train: 1.2485 | val: 1.3071
[Fold 0] Epoch   6/10 | train: 1.2441 | val: 1.2948 <- new best
[Fold 0] Epoch   7/10 | train: 1.2212 | val: 1.2954
[Fold 0] Epoch   8/10 | train: 1.2193 | val: 1.3145
[Fold 0] Epoch   9/10 | train: 1.1718 | val: 1.3059


{'train_losses': [1.389128954905384,
  1.379880094678147,
  1.3621123346892543,
  1.331957739104265,
  1.3025512612840664,
  1.248480316977831,
  1.2441140465766378,
  1.2212020826039824,
  1.2193220186533418,
  1.1717854138440307],
 'val_losses': [1.4202133893966675,
  1.3681927919387817,
  1.295988178253174,
  1.3172677755355835,
  1.3206802606582642,
  1.307084035873413,
  1.2948282480239868,
  1.2954252243041993,
  1.3145376920700074,
  1.3058824300765992],
 'best_model_state': OrderedDict([('input.weight',
               tensor([[-0.0197, -0.0503, -0.1898, -0.1293, -0.1785, -0.1320, -0.1300, -0.2732,
                         0.2886, -0.1520],
                       [ 0.2147, -0.0152,  0.1244, -0.2777,  0.1689,  0.0692, -0.1478,  0.2207,
                        -0.3043, -0.2278],
                       [-0.1486, -0.2436,  0.1922, -0.2670, -0.0717,  0.1644, -0.0845,  0.0654,
                        -0.2349, -0.2083],
                       [-0.2056,  0.2878, -0.0018, -0.0178, -0.196